# مسئلهٔ ۱ — مدل دنباله‌ای ResNet18 + GRU

Baseline قبلی هر فریم را مستقل پیش‌بینی و سپس میانگین‌گیری می‌کرد. این نوت‌بوک ویژگی ۸ فریم مرتب هر ویدئو را به یک GRU می‌دهد تا **ترتیب زمانی** تغییرات صحنه را یاد بگیرد.

برای سبک‌بودن آموزش، ResNet18 fine-tune‌شدهٔ baseline فقط یک‌بار به‌عنوان استخراج‌کنندهٔ ویژگی استفاده می‌شود و سپس GRU روی بردارهای ویژگی آموزش می‌بیند.

In [1]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, f1_score
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18
from torchvision.transforms import v2

DATA_ROOT = Path(r'P:\NexarCollisionData')
MODEL_DIR = DATA_ROOT / 'models'
FRAME_SPLITS_PATH = DATA_ROOT / 'frame_splits.csv'
BASELINE_CHECKPOINT = MODEL_DIR / 'resnet18_frame_baseline.pt'
FEATURE_CACHE = MODEL_DIR / 'resnet18_sequence_features.npz'

RANDOM_SEED = 42
FEATURE_BATCH_SIZE = 32
SEQUENCE_BATCH_SIZE = 32
MAX_EPOCHS = 40
PATIENCE = 7
LEARNING_RATE = 1e-3

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [2]:
frame_splits = pd.read_csv(FRAME_SPLITS_PATH).sort_values(['split', 'video_id', 'frame_order']).reset_index(drop=True)
assert len(frame_splits) == 4800
assert frame_splits.groupby('video_id').size().eq(8).all()
assert frame_splits.groupby('video_id')['split'].nunique().eq(1).all()
assert BASELINE_CHECKPOINT.exists(), 'Run the ResNet18 baseline notebook first.'
frame_splits.groupby(['split', 'label'])['video_id'].nunique()

split       label
train       0        240
            1        240
validation  0         60
            1         60
Name: video_id, dtype: int64

In [3]:
image_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class OrderedFrameDataset(Dataset):
    def __init__(self, table):
        self.table = table.reset_index(drop=True)
    def __len__(self):
        return len(self.table)
    def __getitem__(self, index):
        row = self.table.iloc[index]
        image = Image.open(row.frame_path).convert('RGB')
        return image_transform(image)

if FEATURE_CACHE.exists():
    cache = np.load(FEATURE_CACHE)
    frame_features = cache['features']
    assert len(frame_features) == len(frame_splits)
    print('Loaded cached ResNet features:', frame_features.shape)
else:
    encoder = resnet18(weights=None)
    encoder.fc = nn.Linear(encoder.fc.in_features, 2)
    encoder.load_state_dict(torch.load(BASELINE_CHECKPOINT, map_location=device, weights_only=True)['model_state_dict'])
    encoder.fc = nn.Identity()
    encoder = encoder.to(device).eval()

    batches = []
    loader = DataLoader(OrderedFrameDataset(frame_splits), batch_size=FEATURE_BATCH_SIZE, shuffle=False, num_workers=0)
    with torch.no_grad():
        for images in loader:
            batches.append(encoder(images.to(device)).cpu().numpy())
    frame_features = np.concatenate(batches, axis=0).astype(np.float32)
    np.savez_compressed(FEATURE_CACHE, features=frame_features)
    print('Extracted and cached ResNet features:', frame_features.shape)

Extracted and cached ResNet features: (4800, 512)


In [4]:
frame_splits = frame_splits.assign(feature_index=np.arange(len(frame_splits)))
sequence_records = []
for video_id, group in frame_splits.groupby('video_id', sort=False):
    group = group.sort_values('frame_order')
    sequence_records.append({
        'video_id': video_id,
        'split': group['split'].iloc[0],
        'label': int(group['label'].iloc[0]),
        'features': frame_features[group['feature_index'].to_numpy()],
    })

sequence_table = pd.DataFrame(sequence_records)
train_sequences = sequence_table.query("split == 'train'").reset_index(drop=True)
val_sequences = sequence_table.query("split == 'validation'").reset_index(drop=True)
assert len(train_sequences) == 480 and len(val_sequences) == 120
assert all(features.shape == (8, 512) for features in sequence_table['features'])
print(train_sequences['label'].value_counts().sort_index())
print(val_sequences['label'].value_counts().sort_index())

label
0    240
1    240
Name: count, dtype: int64
label
0    60
1    60
Name: count, dtype: int64


In [5]:
class SequenceDataset(Dataset):
    def __init__(self, table):
        self.table = table.reset_index(drop=True)
    def __len__(self):
        return len(self.table)
    def __getitem__(self, index):
        row = self.table.iloc[index]
        return torch.tensor(row.features), int(row.label), int(row.video_id)

train_loader = DataLoader(SequenceDataset(train_sequences), batch_size=SEQUENCE_BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SequenceDataset(val_sequences), batch_size=SEQUENCE_BATCH_SIZE, shuffle=False)

class GRUVideoClassifier(nn.Module):
    def __init__(self, input_size=512, hidden_size=128):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(hidden_size, 2))
    def forward(self, sequences):
        outputs, _ = self.gru(sequences)
        return self.classifier(outputs[:, -1, :])

model = GRUVideoClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
sum(parameter.numel() for parameter in model.parameters())

246786

In [6]:
def predict_sequences(model, loader):
    model.eval()
    records = []
    with torch.no_grad():
        for sequences, labels, video_ids in loader:
            probabilities = torch.softmax(model(sequences.to(device)), dim=1)[:, 1].cpu().numpy()
            records.extend({
                'video_id': int(video_id), 'label': int(label), 'positive_probability': float(probability)
            } for video_id, label, probability in zip(video_ids, labels, probabilities))
    predictions = pd.DataFrame(records)
    predictions['prediction'] = (predictions['positive_probability'] >= 0.5).astype(int)
    return predictions

best_f1, epochs_without_improvement = -1.0, 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for sequences, labels, _ in train_loader:
        sequences, labels = sequences.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(sequences), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    validation_predictions = predict_sequences(model, val_loader)
    validation_f1 = f1_score(validation_predictions.label, validation_predictions.prediction)
    validation_accuracy = accuracy_score(validation_predictions.label, validation_predictions.prediction)
    metrics = {'epoch': epoch, 'train_loss': total_loss / len(train_loader.dataset), 'validation_f1': validation_f1, 'validation_accuracy': validation_accuracy}
    history.append(metrics)
    print(metrics)

    if validation_f1 > best_f1:
        best_f1, epochs_without_improvement = validation_f1, 0
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch, 'validation_f1': validation_f1}, MODEL_DIR / 'resnet18_gru_sequence.pt')
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

pd.DataFrame(history).to_csv(MODEL_DIR / 'resnet18_gru_history.csv', index=False)

{'epoch': 1, 'train_loss': 0.15424779330690702, 'validation_f1': 0.6557377049180327, 'validation_accuracy': 0.65}
{'epoch': 2, 'train_loss': 0.004277261385383705, 'validation_f1': 0.6, 'validation_accuracy': 0.6333333333333333}
{'epoch': 3, 'train_loss': 0.0010408141805479923, 'validation_f1': 0.6153846153846154, 'validation_accuracy': 0.625}
{'epoch': 4, 'train_loss': 0.0006989182186468194, 'validation_f1': 0.6271186440677966, 'validation_accuracy': 0.6333333333333333}
{'epoch': 5, 'train_loss': 0.0005266032239887863, 'validation_f1': 0.6333333333333333, 'validation_accuracy': 0.6333333333333333}
{'epoch': 6, 'train_loss': 0.0004618979000952095, 'validation_f1': 0.6333333333333333, 'validation_accuracy': 0.6333333333333333}
{'epoch': 7, 'train_loss': 0.0003745441014568011, 'validation_f1': 0.6333333333333333, 'validation_accuracy': 0.6333333333333333}
{'epoch': 8, 'train_loss': 0.000339506931292514, 'validation_f1': 0.6333333333333333, 'validation_accuracy': 0.6333333333333333}
Early 

In [7]:
checkpoint = torch.load(MODEL_DIR / 'resnet18_gru_sequence.pt', map_location=device, weights_only=True)
model.load_state_dict(checkpoint['model_state_dict'])
validation_predictions = predict_sequences(model, val_loader)
validation_predictions.to_csv(MODEL_DIR / 'resnet18_gru_validation_predictions.csv', index=False)

print('Best validation F1:', f"{checkpoint['validation_f1']:.4f}")
print(classification_report(validation_predictions.label, validation_predictions.prediction, digits=4))
validation_predictions.head()

Best validation F1: 0.6557
              precision    recall  f1-score   support

           0     0.6552    0.6333    0.6441        60
           1     0.6452    0.6667    0.6557        60

    accuracy                         0.6500       120
   macro avg     0.6502    0.6500    0.6499       120
weighted avg     0.6502    0.6500    0.6499       120



,video_id,label,positive_probability,prediction
0,14,1,0.828434,1
1,29,1,0.008363,0
2,31,1,0.942512,1
3,32,1,0.900530,1
4,56,1,0.720637,1


## مقایسه با baseline

مدل GRU و baseline باید با split یکسان و در سطح ویدئو مقایسه شوند. اگر GRU بهتر بود، از آن برای نسخهٔ نهایی استفاده می‌کنیم؛ در غیر این صورت baseline ساده‌تر و قابل اتکاتر باقی می‌ماند.